# 02 · Training YOLO11s untuk Deteksi Obstacle Trotoar

Model ini mendeteksi obstacle (bollard, tiang, guiding block, pohon, motor, dll.) yang kemudian diproyeksikan ke peta oleh pipeline geolokasi (`03`). Titik kontak obstacle dengan tanah diambil dari **tepi bawah bounding box**, jadi ketepatan box bagian bawah berpengaruh langsung ke akurasi jarak.

**Output** (dipakai notebook `03`): `dataset_dilans_v8/result_v1/weights/best.pt`

**Ringkasan run terakhir (Sept 2025, dataset v8):** 1980 gambar train, 83 val, 13 kelas. Terbaik di epoch 28 (early stop di 48). mAP50 0.627, mAP50-95 0.329.

**Struktur:** Bagian 1–3 setup dan data · Bagian 4 training (*jangan dijalankan ulang kecuali memang mau retrain*) · Bagian 5 evaluasi dan pemilihan confidence threshold (bisa langsung dijalankan dari `best.pt` di Drive).

## 1. Setup

In [ ]:
# ultralytics dipin ke versi yang dipakai saat training supaya hasil bisa direproduksi.
# Bobot best.pt tetap bisa di-load oleh versi ultralytics yang lebih baru (dipakai di notebook 03).
!pip install -q "ultralytics==8.3.197"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, warnings
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
warnings.filterwarnings("ignore")

DATASET_PATH = "/content/drive/MyDrive/dataset_dilans_v8/"
YAML_PATH    = os.path.join(DATASET_PATH, "data.yaml")
RESULT_DIR   = os.path.join(DATASET_PATH, "result_v1")          # dipakai notebook 03

## 2. Statistik dataset

In [ ]:
with open(YAML_PATH) as f:
    data_cfg = yaml.safe_load(f)
names = data_cfg["names"]
names = names if isinstance(names, list) else [names[k] for k in sorted(names)]
print(f"{len(names)} kelas:", names)

stats = {}
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(DATASET_PATH, split, "images")
    lbl_dir = os.path.join(DATASET_PATH, split, "labels")
    if not os.path.isdir(img_dir):
        continue
    imgs = [f for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    sizes = {Image.open(os.path.join(img_dir, f)).size for f in imgs[:200]}
    counts = np.zeros(len(names), int)
    for f in os.listdir(lbl_dir):
        for line in open(os.path.join(lbl_dir, f)):
            if line.strip():
                counts[int(line.split()[0])] += 1
    stats[split] = counts
    print(f"{split:5s}: {len(imgs)} gambar, ukuran {sizes}")

inst = pd.DataFrame(stats, index=names)
inst

Jumlah instance per kelas penting untuk membaca metrik: kelas dengan sedikit instance di `valid` (mis. < 20) punya mAP yang sangat noisy, dan kelas dengan sedikit instance di `train` cenderung lemah.

## 3. Model dasar

**YOLO11s** (small, ~9.4 juta parameter), pretrained COCO. Varian `s` dipilih sebagai kompromi: lebih akurat dari `n` untuk objek kecil seperti bollard dan tiang, tapi masih cepat untuk dijalankan di Colab.

## 4. Training

> **Jangan dijalankan ulang** kecuali memang mau retrain; `best.pt` hasil run sebelumnya sudah ada di `result_v1/`.

| Parameter | Nilai | Dasar |
|---|---|---|
| `imgsz` | 1024 | Sama dengan ukuran gambar dataset (1024×1024), jadi tidak ada downscale. Objek kecil/tipis (tiang, bollard jauh) butuh resolusi tinggi. |
| `epochs` | 150 | Batas atas; praktis dihentikan oleh early stopping. |
| `patience` | 20 | Berhenti kalau metrik validasi (fitness = 0.1·mAP50 + 0.9·mAP50-95) tidak membaik 20 epoch. Run terakhir: terbaik di epoch 28, berhenti di 48. |
| `batch` | 16 | Muat di GPU A100 40 GB pada 1024 px. Di T4 kemungkinan perlu diturunkan ke 4–8. |
| `optimizer` | AdamW | Konvergensi lebih stabil daripada SGD untuk dataset kecil (~2k gambar) dan learning rate tetap. |
| `lr0` | 0.001 | Learning rate awal yang umum untuk AdamW. |
| `lrf` | 0.01 | Learning rate akhir = `lr0 × lrf` = 1e-5 (jadwal linear). |
| `dropout` | 0.1 | **Catatan:** di Ultralytics, `dropout` hanya berlaku untuk task klasifikasi, jadi untuk deteksi parameter ini tidak berpengaruh. Dibiarkan supaya konsisten dengan run yang sudah ada. |
| `seed` | 42 | Reproducibility. |
| `cache` | True | Gambar disimpan di RAM setelah epoch pertama, sehingga epoch berikutnya jauh lebih cepat. |

In [ ]:
model = YOLO("yolo11s.pt")
results = model.train(
    data=YAML_PATH,
    epochs=150,         # batas atas; dihentikan early stopping
    imgsz=1024,         # = ukuran gambar dataset; objek kecil butuh resolusi tinggi
    patience=20,        # early stopping
    batch=16,           # A100 40 GB; turunkan untuk GPU lebih kecil
    optimizer="AdamW",  # stabil untuk dataset kecil
    lr0=0.001,
    lrf=0.01,           # lr akhir = 1e-5
    dropout=0.1,        # tidak berpengaruh untuk deteksi (hanya klasifikasi); dibiarkan demi konsistensi
    device=0,
    seed=42,
    cache=True,
    name="yolov11_lr1e-3_do10_ep150_datasetv8",
)

In [ ]:
# Salin hasil run ke Drive. Pakai save_dir dari trainer (kalau nama run sudah ada,
# Ultralytics menambahkan akhiran angka, mis. ..._datasetv82, jadi path hard-coded bisa salah).
src = str(model.trainer.save_dir)
shutil.copytree(src, RESULT_DIR, dirs_exist_ok=True)
print(f"{src} -> {RESULT_DIR}")

## 5. Evaluasi

Bisa dijalankan tanpa training ulang: memuat `best.pt` dari Drive dan mengevaluasi di set validasi.

In [ ]:
best = YOLO(os.path.join(RESULT_DIR, "weights", "best.pt"))
metrics = best.val(data=YAML_PATH, imgsz=1024, split="val", plots=True, verbose=False)

b = metrics.box
per_class = pd.DataFrame({
    "class": [best.names[i] for i in b.ap_class_index],
    "precision": b.p, "recall": b.r, "mAP50": b.ap50, "mAP50-95": b.ap,
}).set_index("class")
print(f"ALL  P {b.mp:.3f}  R {b.mr:.3f}  mAP50 {b.map50:.3f}  mAP50-95 {b.map:.3f}")
per_class.sort_values("mAP50").round(3)

### 5.1 Confidence threshold dari kurva F1

Notebook `03` memakai `MIN_CONF` untuk membuang deteksi lemah. Nilai yang paling bisa dipertanggungjawabkan adalah confidence yang **memaksimalkan F1** (keseimbangan precision dan recall) di set validasi, bukan angka tebakan. Nilai di bawah ini yang sebaiknya diisi ke `MIN_CONF` di `03`.

In [ ]:
try:
    f1_all = b.f1_curve.mean(0)            # rata-rata F1 semua kelas, untuk tiap nilai confidence
    best_conf = float(b.px[f1_all.argmax()])
    print(f"Confidence dengan F1 maksimum (semua kelas): {best_conf:.3f}  (F1 = {f1_all.max():.3f})")
    plt.figure(figsize=(7, 4))
    plt.plot(b.px, f1_all); plt.axvline(best_conf, ls="--", c="r")
    plt.xlabel("confidence"); plt.ylabel("F1 (rata-rata kelas)"); plt.grid(True); plt.show()
except AttributeError:
    print("Versi ultralytics ini tidak menyimpan kurva F1 di objek metrik; lihat F1_curve.png di bawah.")

### 5.2 Plot hasil training

In [ ]:
for fname in ["results.png", "F1_curve.png", "confusion_matrix_normalized.png"]:
    path = os.path.join(RESULT_DIR, fname)
    if os.path.exists(path):
        plt.figure(figsize=(14, 8)); plt.imshow(Image.open(path)); plt.axis("off"); plt.title(fname); plt.show()

## Catatan untuk pengembangan

Kelas yang paling relevan untuk trotoar tapi masih lemah (run Sept 2025):

| Kelas | mAP50 | Recall |
|---|---|---|
| Guiding Block | 0.394 | 0.32 |
| pole | 0.423 | 0.32 |
| tree-trunk | 0.493 | 0.47 |
| bollard | 0.798 | 0.76 |

- Recall rendah berarti banyak obstacle terlewat. Tambah data dan anotasi untuk kelas-kelas ini dulu.
- Set validasi hanya 83 gambar (4% data); beberapa kelas hanya 2–9 instance, sehingga metrik per kelas noisy. Proporsi 10–15% lebih ideal.
- Sweep learning rate (dulu ada di notebook lama, belum pernah dijalankan) bisa dilakukan setelah data diperbaiki.

Referensi: Ultralytics YOLO11 docs, train settings: https://docs.ultralytics.com/modes/train/ · validation metrics: https://docs.ultralytics.com/guides/yolo-performance-metrics/